# Micro-Hay Failure Atlas 10

Diagnostica **read-only** della GRU input-only: nessun training e nessuna modifica al dataset o al checkpoint. Per default usa `validation`, preservando il test come holdout confermativo. Il notebook produce viste ortogonali per stato, compartimento, regime, evento, orizzonte, spike, spazio delle fasi, recurrence, spettro del residuo e attivazioni/gate GRU.

Input consigliati: `hay_micro_4c_event_enriched_v2.h5` e il checkpoint `gru_mse.pt` (oppure il vecchio `gru.pt`). Se il checkpoint non e riproducibile sullo schema del dataset, il notebook usa automaticamente un archivio `test_predictions.npz`.

In [ ]:
from pathlib import Path
import subprocess, sys

REPOSITORY_URL = 'https://github.com/Zagred47/LearningSingleCompartiment.git'

def valid_repository(path):
    path = Path(path)
    return ((path / 'pyproject.toml').is_file() and (path / 'src/hay_single_compartment').is_dir()
            and (path / 'notebooks/micro_failure_atlas_10.py').is_file())

candidates = [Path('/kaggle/working/LearningSingleCompartiment')]
candidates += [p.parent for p in Path('/kaggle/working').glob('**/pyproject.toml')]
candidates += [p.parent for p in Path('/kaggle/input').glob('**/pyproject.toml')]
REPO_ROOT = next((p.resolve() for p in candidates if valid_repository(p)), None)
if REPO_ROOT is None:
    base = Path('/kaggle/working/LearningSingleCompartiment_atlas')
    target = base
    suffix = 1
    while target.exists():
        target = Path(f'{base}_{suffix}')
        suffix += 1
    subprocess.check_call(['git', 'clone', '--depth', '1', REPOSITORY_URL, str(target)])
    REPO_ROOT = target.resolve()

src = str(REPO_ROOT / 'src')
if src not in sys.path:
    sys.path.insert(0, src)
import hay_single_compartment
print('Repository:', REPO_ROOT)
print('Package importato da:', Path(hay_single_compartment.__file__).resolve())

## Selezione degli artefatti

La cella seguente mostra i candidati montati. La scoperta automatica preferisce dataset `event_enriched_v2`, checkpoint `gru_mse`/`gru.pt` e `test_predictions.npz`. Se ci sono piu versioni, decommentare le variabili d'ambiente e indicare il path esatto.

In [ ]:
from pathlib import Path
import os

for pattern in ('*.h5', '*.pt', '*.npz'):
    paths = sorted(Path('/kaggle/input').glob(f'**/{pattern}'))
    print(f'\n{pattern}:')
    for path in paths[:30]: print(' ', path)

# Esempio, solo se la scelta automatica non e quella desiderata:
# os.environ['HAY_ATLAS_DATASET'] = '/kaggle/input/NOME-DATASET/hay_micro_4c_event_enriched_v2.h5'
# os.environ['HAY_ATLAS_CHECKPOINT'] = '/kaggle/input/NOME-CHECKPOINT/gru_mse.pt'
# os.environ['HAY_ATLAS_PREDICTIONS'] = '/kaggle/input/NOME-RISULTATI/test_predictions.npz'
# os.environ['HAY_ATLAS_MODEL_KEY'] = 'gru'
# os.environ['HAY_ATLAS_SPLIT'] = 'validation'  # default per checkpoint replay
# os.environ['HAY_ATLAS_PREDICTION_SPLIT'] = 'test'  # vecchi archivi test_predictions
# os.environ['HAY_ATLAS_MAX_TRAJECTORIES'] = '4'  # preflight rapido; lasciare commentato per il report completo
os.environ['HAY_ATLAS_OUTPUT'] = '/kaggle/working/hay_micro_failure_atlas_10'

In [ ]:
import runpy
result = runpy.run_path(str(REPO_ROOT / 'notebooks/micro_failure_atlas_10.py'))
OUTPUT_DIR = Path(result['OUTPUT_DIR'])
ZIP_PATH = OUTPUT_DIR.with_suffix('.zip')
print('Report:', OUTPUT_DIR)
print('ZIP:', ZIP_PATH)

In [ ]:
from IPython.display import Image, display
for name in ('statewise_nrmse.png', 'event_soma_rmse.png', 'horizon_drift.png', 'phase_and_recurrence.png', 'takens_embedding.png', 'soma_spectrum.png', 'gate_saturation_by_event.png'):
    path = OUTPUT_DIR / name
    if path.exists():
        print(name)
        display(Image(filename=str(path)))

In [ ]:
# Download robusto: link normale e fallback Blob (lo ZIP diagnostico e piccolo).
from IPython.display import FileLink, Javascript, display
import base64

display(FileLink(str(ZIP_PATH)))
encoded = base64.b64encode(ZIP_PATH.read_bytes()).decode('ascii')
filename = ZIP_PATH.name
display(Javascript(f'''
const binary = atob('{encoded}');
const bytes = new Uint8Array(binary.length);
for (let i = 0; i < binary.length; i++) bytes[i] = binary.charCodeAt(i);
const blob = new Blob([bytes], {{type: 'application/zip'}});
const url = URL.createObjectURL(blob);
const anchor = document.createElement('a');
anchor.href = url; anchor.download = '{filename}';
document.body.appendChild(anchor); anchor.click(); anchor.remove();
setTimeout(() => URL.revokeObjectURL(url), 60000);
'''))
print('Download avviato:', ZIP_PATH, f'({ZIP_PATH.stat().st_size / 2**20:.1f} MiB)')